In [5]:
import sys, os, math
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_ROOT)
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from guided_diffusion.data import GaussianCircleDataset
from guided_diffusion.denoiser import MLPDenoiser
from guided_diffusion.guidance import guided_sample
from guided_diffusion.verifiers import HalfPlaneVerifier, TargetPointVerifier
from guided_diffusion.metrics import compliance_rate, mode_coverage, modes_covered

ROOT    = Path('..').resolve()
FIG_DIR = ROOT / 'figures'
FIG_DIR.mkdir(exist_ok=True)
CKPT    = ROOT / 'images/ddpm_model.pt'
assert CKPT.exists(), f'Run diffusion.py first: {CKPT}'

plt.rcParams.update({'figure.facecolor':'#0d0d0d','axes.facecolor':'#0d0d0d',
    'axes.edgecolor':'#333','xtick.color':'#666','ytick.color':'#666',
    'text.color':'white','axes.labelcolor':'#aaa','figure.titlesize':13})
SCATTER_KW = dict(s=4, alpha=0.55, linewidths=0)
print('Setup complete.')

Setup complete.


In [6]:
ckpt  = torch.load(CKPT, map_location='cpu')
model = MLPDenoiser(input_dim=ckpt['input_dim'])
model.load_state_dict(ckpt['model_state'])
model.eval()
dataset   = GaussianCircleDataset()
real_pts  = dataset.points.numpy()
K         = dataset.n_clusters
angles    = np.linspace(0, 2*math.pi, K, endpoint=False)
MODE_CTRS = np.stack([dataset.radius*np.cos(angles), dataset.radius*np.sin(angles)], axis=1)
HPV = HalfPlaneVerifier(dim=2, axis=0, temperature=1.0)
TPV = TargetPointVerifier(target=torch.tensor([2.0, 0.0]), sigma=1.0)
REGION = lambda x: x[:, 0] > 0
print(f'Model loaded. {K} modes. MODE_CTRS: {MODE_CTRS.shape}')

Model loaded. 8 modes. MODE_CTRS: (8, 2)


---
## Experiment 1 — Unguided Samples

In [9]:
torch.manual_seed(0)
pts_ug = guided_sample(model, [], n_samples=2000, input_dim=2, w=0).numpy()
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Experiment 1 — Unguided Samples  (2 000 points)')
axes[0].scatter(pts_ug[:,0], pts_ug[:,1], c='#4fc3f7', **SCATTER_KW)
axes[0].set_title('Generated'); axes[0].set_aspect('equal')
axes[0].set_xlim(-4.5,4.5); axes[0].set_ylim(-4.5,4.5)
axes[1].scatter(real_pts[:,0],real_pts[:,1],c='#555',s=1,alpha=0.2,linewidths=0,label='real')
axes[1].scatter(pts_ug[:,0],pts_ug[:,1],c='#4fc3f7',**SCATTER_KW,label='generated')
axes[1].set_title('Generated vs Real'); axes[1].set_aspect('equal')
axes[1].set_xlim(-4.5,4.5); axes[1].set_ylim(-4.5,4.5)
axes[1].legend(markerscale=3, fontsize=8)
plt.tight_layout()
fig.savefig('/Users/ameygupta/sop_task/images/exp1_unguided.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'compliance={compliance_rate(pts_ug,REGION):.3f}  coverage={mode_coverage(pts_ug,MODE_CTRS):.3f}  modes={modes_covered(pts_ug,MODE_CTRS)}/{K}')

compliance=0.475  coverage=1.000  modes=8/8


/var/folders/8j/b2_gp2kd6nj1qs129s9vnh0h0000gn/T/ipykernel_84666/1667979136.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Experiment 2 — Single Verifier Weight Sweep

In [10]:
WEIGHTS = [0, 1, 3, 10]
COLORS  = ['#4fc3f7','#81c784','#ffb74d','#ff7043']
sweep = {}
for w in WEIGHTS:
    torch.manual_seed(0)
    sweep[w] = guided_sample(model,[HPV],n_samples=2000,input_dim=2,w=w).numpy()
    print(f'  w={w}  compliance={compliance_rate(sweep[w],REGION):.3f}')
fig, axes = plt.subplots(1,4,figsize=(20,5))
fig.suptitle('Experiment 2 — HalfPlane Verifier Sweep  (x>0 preferred)')
for ax,w,c in zip(axes,WEIGHTS,COLORS):
    pts=sweep[w]
    ax.scatter(pts[:,0],pts[:,1],color=c,**SCATTER_KW)
    ax.set_title(f'w={w}\ncompliance={compliance_rate(pts,REGION):.2f}')
    ax.set_xlim(-5,5); ax.set_ylim(-5,5); ax.set_aspect('equal')
    ax.axvline(0,color='#555',lw=0.8,ls='--')
plt.tight_layout()
fig.savefig('/Users/ameygupta/sop_task/images/exp2_weight_sweep.png',dpi=150,bbox_inches='tight',facecolor=fig.get_facecolor())
plt.show()

  w=0  compliance=0.475
  w=1  compliance=0.979
  w=3  compliance=1.000
  w=10  compliance=1.000


/var/folders/8j/b2_gp2kd6nj1qs129s9vnh0h0000gn/T/ipykernel_84666/2622147187.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Experiment 3 — Compliance vs Weight  (error bars)

In [11]:
EXP_W = [0,0.5,1,2,3,5,7,10]
SEEDS = [0,1,2,3,4]
comp_results = {w:[] for w in EXP_W}
for w in EXP_W:
    for s in SEEDS:
        torch.manual_seed(s)
        pts = guided_sample(model,[HPV],n_samples=1000,input_dim=2,w=w).numpy()
        comp_results[w].append(compliance_rate(pts,REGION))
    print(f'  w={w}  {np.mean(comp_results[w]):.3f} ± {np.std(comp_results[w]):.3f}')
means = np.array([np.mean(comp_results[w]) for w in EXP_W])
stds  = np.array([np.std( comp_results[w]) for w in EXP_W])
fig, ax = plt.subplots(figsize=(9,5))
ax.plot(EXP_W,means,'o-',color='#4fc3f7',lw=2,ms=6,label='mean')
ax.fill_between(EXP_W,means-stds,means+stds,color='#4fc3f7',alpha=0.25,label='±1 std')
ax.axhline(0.5,color='#555',lw=1,ls='--',label='random baseline')
ax.set_xlabel('Guidance weight  w'); ax.set_ylabel('Compliance rate')
ax.set_title('Experiment 3 — Compliance vs Guidance Weight')
ax.set_ylim(0,1.05); ax.legend(fontsize=9)
plt.tight_layout()
fig.savefig('/Users/ameygupta/sop_task/images/exp3_compliance_vs_weight.png',dpi=150,bbox_inches='tight',facecolor=fig.get_facecolor())
plt.show()

  w=0  0.478 ± 0.017
  w=0.5  0.846 ± 0.005
  w=1  0.987 ± 0.003
  w=2  1.000 ± 0.000
  w=3  1.000 ± 0.000
  w=5  1.000 ± 0.000
  w=7  1.000 ± 0.000
  w=10  1.000 ± 0.000


/var/folders/8j/b2_gp2kd6nj1qs129s9vnh0h0000gn/T/ipykernel_84666/1141509477.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Experiment 4 — Diversity vs Weight

In [12]:
div_results   = {w:[] for w in EXP_W}
nmode_results = {w:[] for w in EXP_W}
for w in EXP_W:
    for s in SEEDS:
        torch.manual_seed(s)
        pts = guided_sample(model,[HPV],n_samples=1000,input_dim=2,w=w).numpy()
        div_results[w].append(mode_coverage(pts,MODE_CTRS))
        nmode_results[w].append(modes_covered(pts,MODE_CTRS))
    print(f'  w={w}  cov={np.mean(div_results[w]):.3f}  modes={np.mean(nmode_results[w]):.1f}')
div_means   = np.array([np.mean(div_results[w])   for w in EXP_W])
div_stds    = np.array([np.std( div_results[w])   for w in EXP_W])
nmode_means = np.array([np.mean(nmode_results[w]) for w in EXP_W])
fig, axes = plt.subplots(1,2,figsize=(14,5))
fig.suptitle('Experiment 4 — Diversity vs Guidance Weight')
axes[0].plot(EXP_W,div_means,'o-',color='#a5d6a7',lw=2,ms=6)
axes[0].fill_between(EXP_W,div_means-div_stds,div_means+div_stds,color='#a5d6a7',alpha=0.25)
axes[0].set_xlabel('w'); axes[0].set_ylabel('Normalised entropy  H/log(K)')
axes[0].set_title('Mode-coverage entropy'); axes[0].set_ylim(0,1.05)
axes[1].plot(EXP_W,nmode_means,'s-',color='#ce93d8',lw=2,ms=6)
axes[1].axhline(K,color='#555',lw=1,ls='--',label=f'max ({K} modes)')
axes[1].set_xlabel('w'); axes[1].set_ylabel('Modes covered')
axes[1].set_title('Number of modes covered'); axes[1].set_ylim(0,K+0.5)
axes[1].legend(fontsize=9)
plt.tight_layout()
fig.savefig('/Users/ameygupta/sop_task/images/exp4_diversity_vs_weight.png',dpi=150,bbox_inches='tight',facecolor=fig.get_facecolor())
plt.show()

  w=0  cov=1.000  modes=8.0
  w=0.5  cov=0.990  modes=8.0
  w=1  cov=0.977  modes=8.0
  w=2  cov=0.969  modes=8.0
  w=3  cov=0.964  modes=8.0
  w=5  cov=0.957  modes=7.0
  w=7  cov=0.947  modes=7.0
  w=10  cov=0.929  modes=5.0


/var/folders/8j/b2_gp2kd6nj1qs129s9vnh0h0000gn/T/ipykernel_84666/3308042161.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Experiment 5 — Pareto Plot

In [13]:
cx = [np.mean(comp_results[w]) for w in EXP_W]
cy = [np.mean(div_results[w])  for w in EXP_W]
ex = [np.std( comp_results[w]) for w in EXP_W]
ey = [np.std( div_results[w])  for w in EXP_W]
fig, ax = plt.subplots(figsize=(8,6))
sc = ax.scatter(cx,cy,c=EXP_W,cmap='plasma',s=90,zorder=5,edgecolors='white',linewidths=0.5)
ax.errorbar(cx,cy,xerr=ex,yerr=ey,fmt='none',color='#555',lw=0.8,capsize=3,zorder=4)
ax.plot(cx,cy,'--',color='#555',lw=1,zorder=3)
for w,x,y in zip(EXP_W,cx,cy):
    ax.annotate(f'w={w}',(x,y),xytext=(6,4),textcoords='offset points',fontsize=8,color='#aaa')
cbar=fig.colorbar(sc,ax=ax,label='Guidance weight  w')
cbar.ax.yaxis.label.set_color('white'); cbar.ax.tick_params(colors='#666')
ax.set_xlabel('Compliance rate'); ax.set_ylabel('Diversity (normalised entropy)')
ax.set_title('Experiment 5 — Pareto Frontier: Compliance vs Diversity')
ax.set_xlim(0.3,1.05); ax.set_ylim(0,1.05)
plt.tight_layout()
fig.savefig('/Users/ameygupta/sop_task/images/exp5_pareto.png',dpi=150,bbox_inches='tight',facecolor=fig.get_facecolor())
plt.show()

/var/folders/8j/b2_gp2kd6nj1qs129s9vnh0h0000gn/T/ipykernel_84666/804694387.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Experiment 6 — Two Verifiers  (4×4 grid)

In [14]:
W1V = [0,1,3,10]  # HalfPlane
W2V = [0,1,3,10]  # TargetPoint
grid = {}
for w1 in W1V:
    for w2 in W2V:
        class _Combo:
            def grad_log_value(self,x):
                g = torch.zeros_like(x)
                if w1>0: g = g + w1*HPV.grad_log_value(x)
                if w2>0: g = g + w2*TPV.grad_log_value(x)
                return g
        torch.manual_seed(0)
        grid[(w1,w2)] = guided_sample(model,[_Combo()],n_samples=500,input_dim=2,w=1).numpy()
print('Grid done.')

Grid done.


In [15]:
fig = plt.figure(figsize=(20,20))
fig.suptitle('Experiment 6 — Two Verifiers\nrows: w₁ (HalfPlane)   cols: w₂ (TargetPoint)',y=1.01)
gs  = gridspec.GridSpec(4,4,figure=fig,hspace=0.35,wspace=0.25)
tgt = np.array([2.,0.])
for i,w1 in enumerate(W1V):
    for j,w2 in enumerate(W2V):
        ax=fig.add_subplot(gs[i,j])
        pts=grid[(w1,w2)]
        ax.scatter(pts[:,0],pts[:,1],c='#4fc3f7',s=5,alpha=0.6,linewidths=0)
        ax.scatter(*tgt,marker='*',s=180,c='#ff7043',zorder=10)
        ax.axvline(0,color='#444',lw=0.7,ls='--')
        cr=compliance_rate(pts,REGION)
        ax.set_title(f'w₁={w1}, w₂={w2}\ncr={cr:.2f}',fontsize=8,pad=4)
        ax.set_xlim(-5,5); ax.set_ylim(-5,5); ax.set_aspect('equal')
        ax.set_xticks([]); ax.set_yticks([])
        if i==3: ax.set_xlabel(f'w₂={w2}',fontsize=8)
        if j==0: ax.set_ylabel(f'w₁={w1}',fontsize=8)
plt.tight_layout()
fig.savefig('/Users/ameygupta/sop_task/images/exp6_two_verifiers_grid.png',dpi=130,bbox_inches='tight',facecolor=fig.get_facecolor())
plt.show()

/var/folders/8j/b2_gp2kd6nj1qs129s9vnh0h0000gn/T/ipykernel_84666/2149914483.py:18: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/var/folders/8j/b2_gp2kd6nj1qs129s9vnh0h0000gn/T/ipykernel_84666/2149914483.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Summary

In [17]:
print(f'{"w":>6} | {"compliance":>12} | {"diversity":>10} | {"modes":>6}')
print('-'*44)
for w in EXP_W:
    print(f'{w:>6.1f} | {np.mean(comp_results[w]):>12.3f} | {np.mean(div_results[w]):>10.3f} | {np.mean(nmode_results[w]):>6.1f}')
print(f'\nFigures → {FIG_DIR}')
[print(f'  {p.name}') for p in sorted(FIG_DIR.glob('*.png'))]

     w |   compliance |  diversity |  modes
--------------------------------------------
   0.0 |        0.478 |      1.000 |    8.0
   0.5 |        0.846 |      0.990 |    8.0
   1.0 |        0.987 |      0.977 |    8.0
   2.0 |        1.000 |      0.969 |    8.0
   3.0 |        1.000 |      0.964 |    8.0
   5.0 |        1.000 |      0.957 |    7.0
   7.0 |        1.000 |      0.947 |    7.0
  10.0 |        1.000 |      0.929 |    5.0

Figures → /Users/ameygupta/sop_task/figures
  exp1_unguided.png
  exp2_weight_sweep.png


[None, None]